# Read 1001 RNA

In [1]:
import polars as pl

In [4]:
import numpy as np


def filter_cv(df: pl.DataFrame, threshold: float=0.1):
    # Calculate the coefficient of variation (CV)
    df = df.fill_null(0.0)
    df = df.fill_nan(0.0)
    np_df_exclude = df.select(pl.selectors.float()).to_numpy()
    
    mean_values = np.mean(np_df_exclude, axis=1)
    cv = np.where(mean_values != 0, np.std(np_df_exclude, axis=1) / mean_values, 0)

    # Filter out features where CV < threshold
    filtered_df = df.filter(cv >= threshold)

    return filtered_df

In [8]:
path_GSE80744 = "/mnt/bank/1001/transcriptomes/GSE80744/my/TPM_GSE80744.csv"
path_GSE54680 = "/mnt/bank/1001/transcriptomes/GSE54680/my/GSE54680_TPM.csv"
path_GSE43858 = ""

In [10]:
mat_gse80744 = pl.read_csv(path_GSE80744)
print(mat_gse80744.shape)
# mat_gse80744 = filter_cv(mat_gse80744)
print(mat_gse80744)

(32833, 729)
shape: (32_833, 729)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Gene_ID   ┆ TPM_SRR34 ┆ TPM_SRX17 ┆ TPM_SRX17 ┆ … ┆ TPM_SRX17 ┆ TPM_SRX17 ┆ TPM_SRX17 ┆ TPM_SRX1 │
│ ---       ┆ 65232     ┆ 34407     ┆ 34408     ┆   ┆ 35130     ┆ 35131     ┆ 35132     ┆ 735133   │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ AT1G01010 ┆ 20.339516 ┆ 9.365095  ┆ 4.737331  ┆ … ┆ 6.25583   ┆ 2.570216  ┆ 7.215608  ┆ 7.080718 │
│ AT1G01020 ┆ 3.671522  ┆ 11.747793 ┆ 8.05168   ┆ … ┆ 7.286584  ┆ 12.790717 ┆ 8.150579  ┆ 16.17857 │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ 3        │
│ AT1G01030 ┆ 5.810495  ┆ 1.510615  ┆ 0.636015  ┆ … ┆ 7.7

In [11]:
mat_GSE54680 = pl.read_csv(path_GSE54680)
print(mat_GSE54680.shape)
# mat_GSE54680 = filter_cv(mat_GSE54680)
print(mat_GSE54680)

(32833, 324)
shape: (32_833, 324)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Gene_ID   ┆ TPM_SRX46 ┆ TPM_SRX46 ┆ TPM_SRX46 ┆ … ┆ TPM_SRX46 ┆ TPM_SRX46 ┆ TPM_SRX46 ┆ TPM_SRX4 │
│ ---       ┆ 5187      ┆ 5188      ┆ 5189      ┆   ┆ 5506      ┆ 5507      ┆ 5508      ┆ 65509    │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ AT1G01010 ┆ 7.564866  ┆ 0.0       ┆ 2.551706  ┆ … ┆ 2.41094   ┆ 2.080416  ┆ 3.950842  ┆ 8.067878 │
│ AT1G01020 ┆ 19.834038 ┆ 5.972422  ┆ 7.47688   ┆ … ┆ 8.211875  ┆ 4.739375  ┆ 7.992414  ┆ 0.0      │
│ AT1G01030 ┆ 3.087381  ┆ 1.688928  ┆ 2.109916  ┆ … ┆ 0.924016  ┆ 0.488509  ┆ 2.488406  ┆ 0.0      │
│ AT1G01040 ┆ 8.857671  ┆ 3.829896  ┆ 1.926939  ┆ … ┆ 2.1

## Join

In [12]:
mat_join = mat_gse80744.join(mat_GSE54680, on="Gene_ID")
print(mat_join)

shape: (32_833, 1_052)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Gene_ID   ┆ TPM_SRR34 ┆ TPM_SRX17 ┆ TPM_SRX17 ┆ … ┆ TPM_SRX46 ┆ TPM_SRX46 ┆ TPM_SRX46 ┆ TPM_SRX4 │
│ ---       ┆ 65232     ┆ 34407     ┆ 34408     ┆   ┆ 5506      ┆ 5507      ┆ 5508      ┆ 65509    │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ AT1G01010 ┆ 20.339516 ┆ 9.365095  ┆ 4.737331  ┆ … ┆ 2.41094   ┆ 2.080416  ┆ 3.950842  ┆ 8.067878 │
│ AT1G01020 ┆ 3.671522  ┆ 11.747793 ┆ 8.05168   ┆ … ┆ 8.211875  ┆ 4.739375  ┆ 7.992414  ┆ 0.0      │
│ AT1G01030 ┆ 5.810495  ┆ 1.510615  ┆ 0.636015  ┆ … ┆ 0.924016  ┆ 0.488509  ┆ 2.488406  ┆ 0.0      │
│ AT1G01040 ┆ 28.866271 ┆ 15.484726 ┆ 19.357944 ┆ … ┆ 2.122822  ┆ 3.

In [15]:
mat_join_f = filter_cv(mat_join, 0.1)
print(mat_join_f.shape)

(32489, 1052)


/tmp/ipykernel_681804/3377951723.py:11: RuntimeWarning: invalid value encountered in divide
  cv = np.where(mean_values != 0, np.std(np_df_exclude, axis=1) / mean_values, 0)


# TMP

In [2]:
micg_SRP173393 = pl.read_parquet("/mnt/bank/scPlantDB/ath/mic_g_init/results/SRP173393.parquet")
print(micg_SRP173393)

shape: (13_252, 6_958)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ obs_names ┆ AT1G01020 ┆ AT1G01050 ┆ AT1G01060 ┆ … ┆ AthLNC005 ┆ AthLNC009 ┆ AthLNC017 ┆ AthLNC02 │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ 219       ┆ 280       ┆ 659       ┆ 0465     │
│ str       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ SRX512861 ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ 5@@_AAACC ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ TGAGAGTAA ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ GG-…      ┆           ┆           ┆           ┆   ┆           ┆   